<a href="https://colab.research.google.com/github/uzair848/AI/blob/main/week8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re

class Expr:
    def __init__(self, op, *args):
        self.op = op  # Predicate name or operator (e.g., 'Fever', '=>')
        self.args = list(args)

    def __repr__(self):
        if self.op == '=>':
            premises, conclusion = self.args[0], self.args[1]
            return f"{' & '.join(map(str, premises))} => {conclusion}"
        elif not self.args:
            return str(self.op)
        return f"{self.op}({', '.join(map(str, self.args))})"

    def __eq__(self, other):
        return isinstance(other, Expr) and self.op == other.op and self.args == other.args

    def __hash__(self):
        return hash((self.op, tuple(self.args)))

def is_variable(x):
    return isinstance(x, str) and x.islower()

# --- Parsing Helper Functions ---
def parse_expr(s):
    s = s.replace(" ", "")
    if '=>' in s:
        lhs, rhs = s.split('=>')
        premises = [parse_atom(p) for p in lhs.split('&')] if lhs else []
        conclusion = parse_atom(rhs)
        return Expr('=>', premises, conclusion)
    return parse_atom(s)

def parse_atom(s):
    # Perfect match for Predicate(arg1,arg2,...)
    match = re.match(r'^([A-Za-z0-9_]+)\(([^)]*)\)$', s)
    if match:
        pred, args_str = match.groups()
        args = args_str.split(',') if args_str else []
        return Expr(pred, *args)
    return Expr(s)


# --- FOL Operations ---
var_counter = 0
def standardize_variables(rule):
    global var_counter
    var_map = {}

    def remap(arg):
        global var_counter
        if is_variable(arg):
            if arg not in var_map:
                var_counter += 1
                var_map[arg] = f"{arg}_{var_counter}"
            return var_map[arg]
        return arg

    def process_expr(expr):
        return Expr(expr.op, *[remap(a) for a in expr.args])

    premises = [process_expr(p) for p in rule.args[0]]
    conclusion = process_expr(rule.args[1])
    return Expr('=>', premises, conclusion)

def subst(theta, expr):
    if isinstance(expr, Expr):
        return Expr(expr.op, *[subst(theta, a) for a in expr.args])
    return theta.get(expr, expr)

def unify(x, y, theta=None):
    if theta is None: theta = {}
    if theta is False: return False
    elif x == y: return theta
    elif is_variable(x): return unify_var(x, y, theta)
    elif is_variable(y): return unify_var(y, x, theta)
    elif isinstance(x, Expr) and isinstance(y, Expr):
        if x.op != y.op or len(x.args) != len(y.args): return False
        for a, b in zip(x.args, y.args):
            theta = unify(a, b, theta)
        return theta
    else: return False

def unify_var(var, val, theta):
    if var in theta: return unify(theta[var], val, theta)
    elif val in theta: return unify(var, theta[val], theta)
    else:
        theta[var] = val
        return theta

def match_premises(premises, kb, theta):
    if not premises:
        yield theta
        return
    first, rest = premises[0], premises[1:]
    for fact in kb:
        subst_first = subst(theta, first)
        new_theta = unify(subst_first, fact, theta.copy())
        if new_theta is not False:
            yield from match_premises(rest, kb, new_theta)

# --- The Core FOL-FC-ASK Algorithm with Output Logging ---
def fol_fc_ask(kb, alpha):
    """
    Asks the KB if alpha is true using Forward Chaining.
    Outputs rules, initial facts, and newly derived facts.
    """
    facts = {item for item in kb if item.op != '=>'}
    rules = [item for item in kb if item.op == '=>']

    print("\n" + "="*40)
    print("‱ INITIAL KNOWLEDGE BASE DETECTED")
    print("="*40)
    print("‹ Rules:")
    for r in rules:
        print(f"  • {r}")
    print("\n‹ Initial Facts:")
    for f in sorted(facts, key=lambda x: str(x)):
        print(f"  • {f}")
    print("="*40)
    print("\n⚙’ RUNNING INFERENCE...")

    step = 1
    while True:
        new = set()
        for rule in rules:
            standard_rule = standardize_variables(rule)
            premises, conclusion = standard_rule.args[0], standard_rule.args[1]

            for theta in match_premises(premises, facts, {}):
                q_prime = subst(theta, conclusion)

                # Check if q_prime is completely new
                if not any(unify(q_prime, f) is not False for f in facts | new):
                    new.add(q_prime)
                    print(f"  [Step {step}] Triggered Rule [{rule}] -> Derived Fact: {q_prime}")
                    step += 1

                    phi = unify(q_prime, alpha)
                    if phi is not False:
                        facts.update(new)
                        return phi, facts, rules

        if not new:
            break
        facts.update(new)

    return False, facts, rules

# --- Interactive User Loop ---
if __name__ == "__main__":
    print("=== First-Order Logic Forward Chaining Engine ===")
    print("Guidelines:")
    print(" - Constants & Predicates must start with an UPPERCASE letter (e.g., John, Fever).")
    print(" - Variables must start with a lowercase letter (e.g., x, y).")
    print(" - Rules use standard implication syntax: Premise1(x) & Premise2(x) => Conclusion(x)")
    print("=" * 50)

    kb = []

    print("\n[Step 1] Enter your Knowledge Base (Facts & Rules).")
    print("You can paste multiple items separated by spaces on a single line.")
    print("Press Enter on an empty line when finished:")

    raw_input = ""
    while True:
        line = input("> ").strip()
        if not line:
            break
        raw_input += " " + line

    statements = re.findall(r'[^\s]+', raw_input)
    for stmt in statements:
        try:
            kb.append(parse_expr(stmt))
        except Exception as e:
            print(f"Malformed statement error on '{stmt}': {e}")

    print("\n[Step 2] Enter the target Query sentence you want to prove:")
    while True:
        query_str = input("Query > ").strip()
        if query_str:
            try:
                query = parse_expr(query_str)
                break
            except Exception as e:
                print(f"Malformed query error: {e}.")

    # Run Algorithm
    result, final_facts, final_rules = fol_fc_ask(kb, query)

    print("\n" + "="*40)
    print("⎁ FINAL EXECUTION SUMMARY")
    print("="*40)
    if result is not False:
        print(" STATUS: ✔ SUCCESS (Query was proven true)")
        print(f" Variable Bindings (̆): {result}")
    else:
        print(" STATUS: ✘ FAILURE (Query cannot be derived)")

    print("\n‑ Final Complete List of Facts in KB:")
    for f in sorted(final_facts, key=lambda x: str(x)):
        print(f"  • {f}")
    print("="*40)

=== First-Order Logic Forward Chaining Engine ===
Guidelines:
 - Constants & Predicates must start with an UPPERCASE letter (e.g., John, Fever).
 - Variables must start with a lowercase letter (e.g., x, y).
 - Rules use standard implication syntax: Premise1(x) & Premise2(x) => Conclusion(x)

[Step 1] Enter your Knowledge Base (Facts & Rules).
You can paste multiple items separated by spaces on a single line.
Press Enter on an empty line when finished:
> Fever(John) SoreThroat(John) MuscleAches(John) Fever(x)&SoreThroat(x)=>InfectionSigns(x) InfectionSigns(x)&MuscleAches(x)=>TestFlu(x)
> 

[Step 2] Enter the target Query sentence you want to prove:
Query > TestFlu(John)

‱ INITIAL KNOWLEDGE BASE DETECTED
‹ Rules:
  • Fever(x) & SoreThroat(x) => InfectionSigns(x)
  • InfectionSigns(x) & MuscleAches(x) => TestFlu(x)

‹ Initial Facts:
  • Fever(John)
  • MuscleAches(John)
  • SoreThroat(John)

⚙’ RUNNING INFERENCE...
  [Step 1] Triggered Rule [Fever(x) & SoreThroat(x) => InfectionSigns(x)]